In [1]:
######################################################################################################################
# Notebook: 03_Curacion_Corpus_RAG
# Autor del código: Vladimir Molleapasa Gutierrez
# Fecha de generación: 13/01/2026
# Código generado con asistencia de ChatGPT 5.2 Thinking
# Prompt original: "Adjunto 3 JSONL (NANDINA, RGI y Notas). Genera un notebook Jupyter llamado 03_Curacion_Corpus_RAG 
#                  que los cargue desde ...\Código\data\processed\, normalice esquema 
#                  (doc_id,tipo,codigo,titulo,texto,fuente,version,idioma,pagina_inicio,pagina_fin), 
#                  deduplique NANDINA (solo 8 dígitos), corrija duplicados en RGI (reglas 1–6 + contexto), 
#                  unifique con Notas y exporte corpus_rag_v1.jsonl + run_metadata (hashes SHA-256) + summary.csv (QA). 
#                  Documenta todo para reproducibilidad."
# Autor del prompt: Vladimir Molleapasa Gutierrez
# Ajustes y validación: Vladimir Molleapasa Gutierrez
# Uso académico, con revisión propia del autor.
# Licencia: Uso académico, no comercial.
######################################################################################################################

# =============================================================================
# Propósito:
#   Curar y unificar tres artefactos JSONL (NANDINA, RGI, Notas legales) en un
#   único corpus documental para RAG (Retrieval-Augmented Generation).
#
# Entradas (data/processed):
#   - nandina_corpus.jsonl
#   - arancel2022_rgi.jsonl
#   - arancel2022_notas.jsonl
#
# Salidas (data/processed):
#   - corpus_rag_v1.jsonl
#       Corpus unificado (NANDINA 8 dígitos + RGI + Notas) con doc_id únicos
#       y esquema homogéneo (doc_id, tipo, codigo, titulo, texto, fuente, etc.)
#   - 03_curacion_run_metadata.json
#       Metadatos reproducibles del run: hashes SHA-256 de entradas/salida,
#       versiones de entorno y parámetros de curación.
#   - 03_curacion_summary.csv
#       QA rápido: conteos por tipo, duplicados, métricas de longitud de texto.
#
# Criterios de curación implementados (mínimos y explícitos):
#   1) NANDINA:
#      - Se conserva únicamente nivel 'nandina_8d' (8 dígitos).
#      - Se deduplica por code_digits.
#      - Se filtran filas “ruidosas” típicas (referencias cruzadas o cabeceras).
#   2) RGI:
#      - Se corrige duplicidad de doc_id.
#      - Se separa preámbulo/contexto como 'rgi_contexto' (opcional) y se deja
#        'rgi' sólo para las reglas 1..6 efectivas.
#   3) Notas:
#      - Se mantienen tal cual, normalizando el esquema.
#
# Dependencias:
#   - Solo biblioteca estándar (json, hashlib, platform, statistics, pathlib).
# =============================================================================


In [2]:
# =============================================================================
# Configuración de rutas (repositorio)
# =============================================================================
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

IN_NANDINA = DATA_PROCESSED / "nandina_corpus.jsonl"
IN_RGI     = DATA_PROCESSED / "arancel2022_rgi.jsonl"
IN_NOTAS   = DATA_PROCESSED / "arancel2022_notas.jsonl"

OUT_CORPUS = DATA_PROCESSED / "corpus_rag_v1.jsonl"
OUT_META   = DATA_PROCESSED / "03_curacion_run_metadata.json"
OUT_QA_CSV = DATA_PROCESSED / "03_curacion_summary.csv"

for p in [IN_NANDINA, IN_RGI, IN_NOTAS]:
    assert p.exists(), f"No se encuentra el archivo requerido: {p}"

print("Entradas:")
print(" -", IN_NANDINA)
print(" -", IN_RGI)
print(" -", IN_NOTAS)
print("Salidas:")
print(" -", OUT_CORPUS)
print(" -", OUT_META)
print(" -", OUT_QA_CSV)


Entradas:
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\nandina_corpus.jsonl
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_rgi.jsonl
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\arancel2022_notas.jsonl
Salidas:
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\corpus_rag_v1.jsonl
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\03_curacion_run_metadata.json
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\03_curacion_summary.csv


In [3]:
# =============================================================================
# Utilidades: JSONL, hashing, entorno, QA básico
# =============================================================================
import json
import hashlib
import platform
from datetime import datetime
from collections import Counter
from statistics import mean, median

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Hash SHA-256 del archivo (t trazabilidad reproducible del insumo/salida)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def load_jsonl(path: Path) -> list[dict]:
    """Carga un JSONL en memoria como lista de dicts (1 dict por línea)."""
    records = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON inválido en {path.name}, línea {i}: {e}")
    return records

def write_jsonl(records: list[dict], path: Path) -> None:
    """Escribe un JSONL (una línea JSON por registro), UTF-8."""
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def environment_metadata() -> dict:
    """Metadatos mínimos del entorno para reproducibilidad."""
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "python_version": platform.python_version(),
        "platform": platform.platform(),
    }

def text_len_stats(texts: list[str]) -> dict:
    """Estadísticas descriptivas de longitudes (caracteres) para QA."""
    if not texts:
        return {"n": 0}
    lens = [len(t) for t in texts]
    return {
        "n": len(lens),
        "min": min(lens),
        "max": max(lens),
        "mean": round(mean(lens), 2),
        "median": median(lens),
    }

def find_doc_id_duplicates(records: list[dict]) -> list[str]:
    """Retorna doc_id duplicados (si existen)."""
    ids = [r.get("doc_id") for r in records if r.get("doc_id")]
    c = Counter(ids)
    return sorted([k for k, v in c.items() if v > 1])


In [4]:
# =============================================================================
# Carga de insumos y validación mínima de esquema
# =============================================================================
nandina_raw = load_jsonl(IN_NANDINA)
rgi_raw     = load_jsonl(IN_RGI)
notas_raw   = load_jsonl(IN_NOTAS)

print("Conteos insumo:")
print(" - NANDINA:", len(nandina_raw))
print(" - RGI:", len(rgi_raw))
print(" - Notas:", len(notas_raw))

# Validación mínima de campos esperados (no bloqueante, pero explicita supuestos)
def require_keys(sample: dict, keys: list[str], name: str) -> None:
    missing = [k for k in keys if k not in sample]
    assert not missing, f"Esquema inesperado en {name}: faltan campos {missing}"

require_keys(nandina_raw[0], ["code_digits", "level", "description", "page"], "nandina_corpus.jsonl")
require_keys(rgi_raw[0], ["doc_id", "tipo", "titulo", "texto"], "arancel2022_rgi.jsonl")
require_keys(notas_raw[0], ["doc_id", "tipo", "scope", "titulo", "texto"], "arancel2022_notas.jsonl")

print("Esquemas verificados (mínimo).")


Conteos insumo:
 - NANDINA: 9785
 - RGI: 8
 - Notas: 96
Esquemas verificados (mínimo).


In [5]:
# =============================================================================
# Curación NANDINA (8 dígitos, deduplicación, filtro de ruido)
# =============================================================================
import re

NOISE_PATTERNS = [
    re.compile(r"\bC[oó]digo\s+Designaci[oó]n\b", re.IGNORECASE),  # cabecera “Código Designación...”
]

def is_noise_nandina(rec: dict) -> bool:
    """
    Heurísticas conservadoras para eliminar filas no representativas del código:
      - referencias cruzadas (p.ej. '4805.19.00, 4805.24.00 o ...)')
      - cabeceras incrustadas en la línea
    """
    desc = (rec.get("description") or "").strip()
    line = (rec.get("line_text") or "").strip()

    # caso típico: descripción empieza con coma => referencia a otros códigos
    if desc.startswith(","):
        return True

    # cabeceras incrustadas o texto no normativo
    for pat in NOISE_PATTERNS:
        if pat.search(desc) or pat.search(line):
            return True

    # referencia cruzada: si inmediatamente tras el código hay una coma en la línea original
    # (ej.: "4805.19.00, 4805.24.00 ...")
    code_raw = (rec.get("code_raw") or "").strip()
    if code_raw and (line.startswith(code_raw + ",")):
        return True

    return False

def nandina_score(rec: dict) -> tuple:
    """
    Ranking de candidatos cuando hay duplicados por code_digits.
    Criterio:
      1) preferir no-ruido
      2) preferir presencia de unidad
      3) preferir descripción más larga (más informativa)
    """
    noise = is_noise_nandina(rec)
    unit = 1 if rec.get("unit") else 0
    desc_len = len((rec.get("description") or "").strip())
    return (0 if not noise else -1, unit, desc_len)

def curate_nandina(nandina_records: list[dict]) -> tuple[list[dict], dict]:
    """
    Salida:
      - lista de documentos normalizados tipo 'nandina_8'
      - métricas QA específicas de NANDINA
    """
    # filtrar nivel
    nandina_8 = [r for r in nandina_records if r.get("level") == "nandina_8d"]

    # deduplicar por code_digits eligiendo mejor candidato por score
    best_by_code: dict[str, dict] = {}
    for r in nandina_8:
        code = r["code_digits"]
        if code not in best_by_code or nandina_score(r) > nandina_score(best_by_code[code]):
            best_by_code[code] = r

    curated = []
    removed_noise = 0
    for code, r in best_by_code.items():
        if is_noise_nandina(r):
            removed_noise += 1  # se contabiliza; aún así se conserva (es raro) salvo que se decida lo contrario

        titulo = (r.get("description") or "").strip()
        texto = titulo

        # Contexto mínimo (mejora recuperación sin “inventar” contenido)
        ctx = []
        if r.get("section"):
            ctx.append(f"Sección {r['section']}")
        if r.get("chapter"):
            ctx.append(f"Capítulo {r['chapter']}")
        if ctx:
            texto = f"{texto}. Contexto: " + " / ".join(ctx) + "."

        curated.append({
            "doc_id": f"NANDINA_{code}",
            "tipo": "nandina_8",
            "codigo": code,
            "titulo": titulo,
            "texto": texto,
            "fuente": "NANDINA",
            "version": "Decision_885",
            "idioma": "es",
            "pagina_inicio": r.get("page"),
            "pagina_fin": r.get("page"),
            "section": r.get("section"),
            "chapter": r.get("chapter"),
        })

    # QA NANDINA
    qa = {
        "nandina_input_total": len(nandina_records),
        "nandina_input_8d": len(nandina_8),
        "nandina_output_unique_8d": len(curated),
        "nandina_noise_selected_count": removed_noise,
        "nandina_output_doc_id_duplicates": len(find_doc_id_duplicates(curated)),
    }
    return curated, qa

nandina_curated, qa_nandina = curate_nandina(nandina_raw)
print("NANDINA curada:", len(nandina_curated))
print("QA NANDINA:", qa_nandina)


NANDINA curada: 7644
QA NANDINA: {'nandina_input_total': 9785, 'nandina_input_8d': 7648, 'nandina_output_unique_8d': 7644, 'nandina_noise_selected_count': 0, 'nandina_output_doc_id_duplicates': 0}


In [6]:
# =============================================================================
# Curación RGI (resolver doc_id duplicados y separar contexto)
# =============================================================================
def is_rgi_rule_text(text: str) -> bool:
    """
    Identifica reglas RGI efectivas (1..6) vs contexto/preámbulo.
    Criterio mínimo:
      - Regla 1 efectiva comienza típicamente con '1. Los títulos...'
      - Regla 2 efectiva comienza con '2. a)' en la mayoría de aranceles.
    """
    t = (text or "").lstrip()
    return (
        t.startswith("1. Los títulos")
        or t.startswith("2. a)")
        or t.startswith("3.")
        or t.startswith("4.")
        or t.startswith("5.")
        or t.startswith("6.")
    )

def curate_rgi(rgi_records: list[dict], keep_context: bool = True) -> tuple[list[dict], dict]:
    """
    Curación:
      - Registros con texto no-regla se recodifican como 'rgi_contexto' y doc_id únicos.
      - Registros regla se mantienen como 'rgi' y doc_id se conserva (si es único).
    """
    curated = []
    context_count = 0
    rule_count = 0

    # Reasignación estable de doc_id para evitar colisiones
    context_seq = 0

    for r in rgi_records:
        text = r.get("texto", "")
        if is_rgi_rule_text(text):
            rule_count += 1
            curated.append({
                **r,
                "tipo": "rgi",  # fuerza coherencia
            })
        else:
            context_count += 1
            if keep_context:
                context_seq += 1
                curated.append({
                    **r,
                    "doc_id": f"ARANCEL2022_RGI_CONTEXTO_{context_seq}",
                    "tipo": "rgi_contexto",
                    "ref": r.get("ref") or f"RGI contexto {context_seq}",
                    "titulo": r.get("titulo") or f"Contexto RGI {context_seq}",
                })

    # QA RGI
    dups = find_doc_id_duplicates(curated)
    qa = {
        "rgi_input_total": len(rgi_records),
        "rgi_output_total": len(curated),
        "rgi_rules_kept": rule_count,
        "rgi_context_kept": context_count if keep_context else 0,
        "rgi_output_doc_id_duplicates": len(dups),
        "rgi_doc_id_duplicates": dups[:20],
    }
    return curated, qa

rgi_curated, qa_rgi = curate_rgi(rgi_raw, keep_context=True)
print("RGI curada:", len(rgi_curated))
print("QA RGI:", qa_rgi)


RGI curada: 8
QA RGI: {'rgi_input_total': 8, 'rgi_output_total': 8, 'rgi_rules_kept': 6, 'rgi_context_kept': 2, 'rgi_output_doc_id_duplicates': 0, 'rgi_doc_id_duplicates': []}


In [7]:
# =============================================================================
# Curación Notas (normalización de esquema)
# =============================================================================
def curate_notas(notas_records: list[dict]) -> tuple[list[dict], dict]:
    """
    Normaliza campos y conserva contenido.
    Se garantiza presencia de: doc_id, tipo, texto, fuente, version, idioma.
    """
    curated = []
    for r in notas_records:
        curated.append({
            "doc_id": r["doc_id"],
            "tipo": r["tipo"],                  # nota_seccion / nota_capitulo
            "codigo": r.get("chapter") or r.get("section"),
            "titulo": r.get("titulo"),
            "texto": r.get("texto"),
            "fuente": r.get("fuente"),
            "version": r.get("version", "Arancel_2022"),
            "idioma": r.get("idioma", "es"),
            "pagina_inicio": r.get("pagina_inicio"),
            "pagina_fin": r.get("pagina_fin"),
            "scope": r.get("scope"),
            "section": r.get("section"),
            "chapter": r.get("chapter"),
        })

    qa = {
        "notas_input_total": len(notas_records),
        "notas_output_total": len(curated),
        "notas_output_doc_id_duplicates": len(find_doc_id_duplicates(curated)),
        "counts_by_tipo": dict(Counter([r["tipo"] for r in curated])),
    }
    return curated, qa

notas_curated, qa_notas = curate_notas(notas_raw)
print("Notas curadas:", len(notas_curated))
print("QA Notas:", qa_notas)


Notas curadas: 96
QA Notas: {'notas_input_total': 96, 'notas_output_total': 96, 'notas_output_doc_id_duplicates': 0, 'counts_by_tipo': {'nota_seccion': 9, 'nota_capitulo': 87}}


In [8]:
# =============================================================================
# Unificación, QA global y escritura de artefactos
# =============================================================================
corpus_final = []
corpus_final.extend(nandina_curated)
corpus_final.extend(rgi_curated)
corpus_final.extend(notas_curated)

# QA global
counts = Counter([r.get("tipo") for r in corpus_final])
dup_doc_ids = find_doc_id_duplicates(corpus_final)
texts = [r.get("texto","") for r in corpus_final if isinstance(r.get("texto",""), str)]

qa_global = {
    "records_total": len(corpus_final),
    "counts_by_tipo": dict(counts),
    "doc_id_duplicates_count": len(dup_doc_ids),
    "doc_id_duplicates_sample": dup_doc_ids[:50],
    "texto_length_stats": text_len_stats(texts),
}

# Escritura de corpus
assert qa_global["doc_id_duplicates_count"] == 0, (
    f"Existen doc_id duplicados: {dup_doc_ids[:10]} (ver sample). "
    "Resolver antes de indexar."
)
write_jsonl(corpus_final, OUT_CORPUS)

# Run metadata (reproducibilidad)
run_meta = {
    "notebook": "03_Curación_de_corpus",
    "environment": environment_metadata(),
    "inputs": {
        "nandina_jsonl": {"path": str(IN_NANDINA), "sha256": sha256_file(IN_NANDINA), "records": len(nandina_raw)},
        "rgi_jsonl":     {"path": str(IN_RGI),     "sha256": sha256_file(IN_RGI),     "records": len(rgi_raw)},
        "notas_jsonl":   {"path": str(IN_NOTAS),   "sha256": sha256_file(IN_NOTAS),   "records": len(notas_raw)},
    },
    "outputs": {
        "corpus_jsonl":  {"path": str(OUT_CORPUS), "sha256": sha256_file(OUT_CORPUS), "records": len(corpus_final)},
        "qa_csv":        {"path": str(OUT_QA_CSV)},
    },
    "curation_parameters": {
        "nandina_keep_level": "nandina_8d",
        "nandina_dedup_key": "code_digits",
        "nandina_noise_filters": [
            "desc startswith ',' (cross-reference)",
            "line startswith '<code_raw>,' (cross-reference)",
            "contains 'Código Designación' (header contamination)",
        ],
        "rgi_keep_context": True,
        "rgi_rule_detection": [
            "text startswith '1. Los títulos' OR '2. a)' OR '3.'..'6.'"
        ],
    },
    "qa": {
        "nandina": qa_nandina,
        "rgi": qa_rgi,
        "notas": qa_notas,
        "global": qa_global,
    }
}

OUT_META.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

# QA CSV simple (rápido para Excel)
qa_lines = ["metric,value"]
qa_lines.append(f"records_total,{qa_global['records_total']}")
for k, v in sorted(counts.items()):
    qa_lines.append(f"count_{k},{v}")
qa_lines.append(f"doc_id_duplicates_count,{qa_global['doc_id_duplicates_count']}")
qa_lines.append(f"texto_len_min,{qa_global['texto_length_stats'].get('min','')}")
qa_lines.append(f"texto_len_max,{qa_global['texto_length_stats'].get('max','')}")
qa_lines.append(f"texto_len_mean,{qa_global['texto_length_stats'].get('mean','')}")
qa_lines.append(f"texto_len_median,{qa_global['texto_length_stats'].get('median','')}")

OUT_QA_CSV.write_text("\n".join(qa_lines), encoding="utf-8")

print("=== Curación completada ===")
print("Corpus final:", OUT_CORPUS)
print("Run metadata:", OUT_META)
print("QA summary:", OUT_QA_CSV)
print("QA global:", qa_global)


=== Curación completada ===
Corpus final: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\corpus_rag_v1.jsonl
Run metadata: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\03_curacion_run_metadata.json
QA summary: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\03_curacion_summary.csv
QA global: {'records_total': 7748, 'counts_by_tipo': {'nandina_8': 7644, 'rgi_contexto': 2, 'rgi': 6, 'nota_seccion': 9, 'nota_capitulo': 87}, 'doc_id_duplicates_count': 0, 'doc_id_duplicates_sample': [], 'texto_length_stats': {'n': 7748, 'min': 40, 'max': 17022, 'mean': 104.11, 'median': 59.0}}


In [9]:
# =============================================================================
# Inspección rápida (muestras)
# =============================================================================
import itertools

def head_jsonl(path: Path, n=3):
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in itertools.islice(f, n):
            out.append(json.loads(line))
    return out

print("Muestra corpus (primeras 3 líneas):")
for rec in head_jsonl(OUT_CORPUS, 3):
    print(rec["doc_id"], "|", rec["tipo"], "|", (rec.get("codigo") or ""))
    print((rec.get("texto") or "")[:200], "...\n")


Muestra corpus (primeras 3 líneas):
NANDINA_01012100 | nandina_8 | 01012100
Reproductores de raza pura. Contexto: Sección I / Capítulo 01. ...

NANDINA_01012910 | nandina_8 | 01012910
Para carrera. Contexto: Sección I / Capítulo 01. ...

NANDINA_01012990 | nandina_8 | 01012990
Los demás. Contexto: Sección I / Capítulo 01. ...



In [10]:
# =============================================================================
# Limpieza de texto para indexación (sin alterar el texto original)
# Genera: corpus_rag_v1_index.jsonl con campo "texto_index"
# =============================================================================
import json
import re
import time
import hashlib
from pathlib import Path

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

_CONTEXT_RE = re.compile(r"\bcontexto\s*:\s.*$", flags=re.IGNORECASE | re.DOTALL)

def build_texto_index(texto: str) -> str:
    """
    Limpieza mínima orientada a recuperación léxica (BM25):
    - Elimina el bloque "Contexto: ..." (frecuentemente incluye Sección/Capítulo y números)
    - Normaliza espacios
    Nota: se preserva el 'texto' original para auditoría y RAG.
    """
    if texto is None:
        return ""
    t = str(texto).strip()
    t = _CONTEXT_RE.sub("", t).strip()
    t = re.sub(r"\s+", " ", t).strip()
    return t

OUT_CORPUS_INDEX = DATA_PROCESSED / "corpus_rag_v1_index.jsonl"
OUT_INDEX_META   = DATA_PROCESSED / "03_curacion_index_text_metadata.json"

in_sha = sha256_file(OUT_CORPUS)

n_rows = 0
n_with_context_removed = 0

with open(OUT_CORPUS, "r", encoding="utf-8") as fin, open(OUT_CORPUS_INDEX, "w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        texto_original = obj.get("texto", "")

        texto_index = build_texto_index(texto_original)
        obj["texto_index"] = texto_index  # campo nuevo, no destructivo

        # métrica simple: detecta si hubo recorte por "Contexto:"
        if isinstance(texto_original, str) and re.search(r"\bcontexto\s*:", texto_original, flags=re.IGNORECASE):
            # si había contexto y ahora no está, lo contamos
            if not re.search(r"\bcontexto\s*:", texto_index, flags=re.IGNORECASE):
                n_with_context_removed += 1

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
        n_rows += 1

out_sha = sha256_file(OUT_CORPUS_INDEX)

meta = {
    "timestamp_unix": int(time.time()),
    "input": {"path": str(OUT_CORPUS), "sha256": in_sha},
    "output": {"path": str(OUT_CORPUS_INDEX), "sha256": out_sha},
    "transform": {
        "strategy": "add_field_texto_index",
        "rule": "remove substring from 'Contexto:' to end-of-text; normalize whitespace"
    },
    "counts": {
        "rows_processed": n_rows,
        "rows_where_context_was_removed_estimate": n_with_context_removed
    }
}

with open(OUT_INDEX_META, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("OK: Generado corpus con texto limpio para indexación.")
print(" -", OUT_CORPUS_INDEX)
print(" - meta:", OUT_INDEX_META)
print(" - input_sha256:", in_sha)
print(" - output_sha256:", out_sha)
print(" - rows:", n_rows)


OK: Generado corpus con texto limpio para indexación.
 - C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\corpus_rag_v1_index.jsonl
 - meta: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\03_curacion_index_text_metadata.json
 - input_sha256: 14b6de5438d4e7058f8bcc9a744858e46d65c51a9ae014215374e6cb37a033fc
 - output_sha256: 83768faae816b9d9b33a8fd36b73068d8b5f0b7a186e1c0f5b1c2c27580290f0
 - rows: 7748
